In [45]:
import os
from langgraph import graph
from langgraph.graph import StateGraph,END,START
from typing import Dict,TypedDict,List,Union
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

In [46]:
load_dotenv()

class AgentState(TypedDict):
    messages: List[Union[HumanMessage,AIMessage]]


llm = ChatDeepSeek(model="deepseek-chat", temperature=0)

def process(state:AgentState) -> AgentState:
    response = llm.invoke(state['messages'])
    state['messages'].append(AIMessage(content=response.content))
    print('Curent_state：',state['messages'])
    return state


In [47]:
graph = StateGraph(AgentState)
graph.add_node('process_node', process)
graph.add_edge(START, 'process_node')
graph.add_edge('process_node', END)

agent = graph.compile()

In [50]:
conversation_message = []
while True:
    user_input = input('Enter:')
    if user_input == 'exit':
        break
    conversation_message.append(HumanMessage(content=user_input))
    result = agent.invoke({'messages':conversation_message})
    print('AI:',result['messages'][-1].content)
    conversation_message = result['messages']


Curent_state： [HumanMessage(content='hi\n', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
AI: Hi! How can I help you today?
Curent_state： [HumanMessage(content='hi\n', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='nihao\n', additional_kwargs={}, response_metadata={}), AIMessage(content='你好！有什么我可以帮你的吗？😊', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
AI: 你好！有什么我可以帮你的吗？😊


In [51]:
from langchain_core.messages import HumanMessage, AIMessage

with open("logging.txt", "w", encoding="utf-8") as file:
    file.write("Your Conversation Log:\n")
    for message in conversation_message:
        if isinstance(message, HumanMessage):
            file.write(f"You: {message.content}\n")
        elif isinstance(message, AIMessage):
            file.write(f"AI: {message.content}\n\n")
    file.write("End of Conversation\n")

print("Conversation saved to logging.txt")

Conversation saved to logging.txt
